# 01 · Data cleaning and topic filtering

Takes the raw r/dogecoin posts, drops duplicates and empty posts, joins title and body, and keeps on-topic posts in two layers:
1. **Keyword filter** — regex on terms like `doge`, `dogecoin`, `elon`, `hodl`, `memecoin`.
2. **Semantic filter** — TF-IDF vectors of each post compared (cosine similarity) with a few hand-written reference sentences, so on-topic posts without an exact keyword are kept.

The later cells are the first DeepSeek scoring runs (synchronous, then async) that were rebuilt in `03`.

*Input:* raw Reddit posts (not included, see `data/README.md`) · *Output:* filtered post list

In [ ]:
import os
import pandas as pd
from tqdm import tqdm
from openai import OpenAI

# My DeepSeek API key
client = OpenAI(api_key=os.environ["DEEPSEEK_API_KEY"], base_url="https://api.deepseek.com")

# read Reddit dataset
df = pd.read_csv("one-year-of-doge-on-reddit-posts.csv")

# preprocessing
df = df[df['type'] == 'post']
df['text'] = (df['title'].fillna('') + " " + df['selftext'].fillna('')).str.strip()
df = df.drop_duplicates(subset=['title'])
df = df[df['text'].str.strip() != ""]
df['date'] = pd.to_datetime(df['date'])

# Define DeepSeek fuction
def get_sentiment_arousal(text):
    try:
        prompt = (
            "You are a financial sentiment analyzer. "
            "Given the following Reddit post, output two numeric scores in JSON format: "
            "sentiment (-1 to 1, where -1 is very negative and 1 is very positive), "
            "and arousal (0 to 1, where 1 is very excited and 0 is very calm). "
            f"Post: {text[:1500]}"
        )
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
        )
        output = response.choices[0].message.content
        import re, json
        # JSON
        match = re.search(r'\{.*\}', output, re.DOTALL)
        if match:
            data = json.loads(match.group(0))
            return data.get("sentiment", None), data.get("arousal", None)
        else:
            return None, None
    except Exception as e:
        print("Error:", e)
        return None, None

# score in batch
sentiments, arousals = [], []
for text in tqdm(df['text'], desc="Scoring Reddit posts"):
    s, a = get_sentiment_arousal(text)
    sentiments.append(s)
    arousals.append(a)

df['sentiment'] = sentiments
df['arousal'] = arousals

# cluster the daily sentiment
daily = df.groupby('date')[['sentiment', 'arousal']].mean().reset_index()
daily.to_csv("reddit_daily_sentiment.csv", index=False)

print("✅ Saved reddit_daily_sentiment.csv successfully!")


cleaning

In [ ]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Read dsts
df = pd.read_csv("../data/raw/doge_posts_with_date.csv")
df = df[df['type'] == 'post']

# Delete deplicates
df = df.drop_duplicates(subset=['title'])

# combine title + selftext
df['text'] = (df['title'].fillna('') + " " + df['selftext'].fillna('')).str.strip()

# Delete blank
df = df[df['text'].str.strip() != ""]

# Date
df['date'] = pd.to_datetime(df['date']).dt.date

# Filter
keywords = [
    r'\bdoge\b',
    r'\bdogecoin\b',
    r'\bdoge\s*coin\b',
    r'\bshibadoge\b',
    r'\belon\b',
    r'\bmusk\b',
    r'\bmoon\b',
    r'\bto\s+the\s+moon\b',
    r'\bhodl\b',
    r'\bmemecoin\b',
    r'\bcrypto\b',
    r'\bcoin\b'
]

pattern = re.compile('|'.join(keywords), flags=re.IGNORECASE)
df['keyword_match'] = df['text'].apply(lambda x: bool(pattern.search(x)))

# Keep keywords
df_kw = df[df['keyword_match'] == True].copy()
print(f" Keyword filter retained {len(df_kw)} posts out of {len(df)} total.")

#3. NLP similarity Filter
reference_docs = [
    "dogecoin",
    "doge coin",
    "dogecoin price",
    "elon musk dogecoin tweet",
    "dogecoin community",
    "dogecoin meme",
    "dogecoin pump",
    "doge to the moon",
    "dogecoin holder sentiment"
]

vectorizer = TfidfVectorizer(stop_words='english', max_features=10000)
tfidf_matrix = vectorizer.fit_transform(df_kw['text'].tolist() + reference_docs)

# reference semantic center vector
ref_vec = tfidf_matrix[-len(reference_docs):].mean(axis=0)
ref_vec = np.asarray(ref_vec)

# Similarity
similarities = cosine_similarity(tfidf_matrix[:-len(reference_docs)], ref_vec)

df_kw['semantic_score'] = similarities.flatten()

# Threshold
threshold = 0.08
df_kw['strong_related'] = df_kw['semantic_score'] > threshold

# Output
df_strong = df_kw[df_kw['strong_related'] == True].copy()
df_strong = df_strong[['date', 'title', 'selftext', 'text', 'subreddit.name', 'semantic_score']]
df_strong.rename(columns={'subreddit.name': 'subreddit'}, inplace=True)

df_strong.to_csv("../data/raw/dogecoin_reddit_strong_related.csv", index=False)

print(f"Final Dogecoin-strong-related posts saved: {len(df_strong)}")


/var/folders/1g/jbrl1xp924q5fwsrcgtxwmx80000gn/T/ipykernel_28833/2323522301.py:9: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/doge_posts_with_date.csv")


✅ Keyword filter retained 317342 posts out of 317376 total.
✅ Final Dogecoin-strong-related posts saved: 41590


In [ ]:
import os
import nest_asyncio
import asyncio
import pandas as pd
from tqdm import tqdm
import json
import re
nest_asyncio.apply()

API_KEY = os.environ["DEEPSEEK_API_KEY"]
API_URL = "https://api.deepseek.com/v1/chat/completions"

# Factual
def is_factual(text):
    factual_keywords = [
        'price is', 'market cap', 'trading volume', 'exchange', 'listed on',
        '% change', 'up by', 'down by', 'ranked', 'market data', '24h', '7d'
    ]
    return any(k in text.lower() for k in factual_keywords)

# Prompt to score
def make_prompt(text):
    return f"""
You are an emotion analysis model.
Given the text below, output in JSON format:
{{
  "sentiment": one of ["positive", "neutral", "negative"],
  "arousal": a number between 0 and 1 (how emotionally intense it is)
}}
If the text is purely factual or news-like, output:
{{"sentiment": "neutral", "arousal": 0}}
Text: "{text}"
"""

# DeepSeek API Analysis
async def analyze_text(session, text):
    try:
        payload = {
            "model": "deepseek-chat",
            "messages": [{"role": "user", "content": make_prompt(text)}],
            "temperature": 0
        }
        async with session.post(API_URL, headers={"Authorization": f"Bearer {API_KEY}"}, json=payload) as resp:
            result = await resp.json()
            reply = result["choices"][0]["message"]["content"]
            data = json.loads(re.search(r"\{.*\}", reply, re.S).group(0))
            return data
    except Exception as e:
        return {"sentiment": "error", "arousal": None, "error": str(e)}

# Process in batch
async def batch_process(texts, batch_size=20):
    results = []
    async with aiohttp.ClientSession() as session:
        for i in tqdm(range(0, len(texts), batch_size)):
            batch = texts[i:i+batch_size]
            tasks = [analyze_text(session, t) for t in batch]
            batch_results = await asyncio.gather(*tasks)
            results.extend(batch_results)
            await asyncio.sleep(2)
    return results

async def run_sentiment_pipeline_async(df):
    df = df.copy()
    df['is_factual'] = df['text'].apply(is_factual)
    df_nonfact = df[df['is_factual'] == False].reset_index(drop=True)

    print(f"Processing {len(df_nonfact)} emotional texts via DeepSeek API...")

    results = await batch_process(df_nonfact['text'].tolist())

    df_results = pd.DataFrame(results)
    df_nonfact = pd.concat([df_nonfact, df_results], axis=1)
    return df_nonfact

df = pd.read_csv("../data/raw/dogecoin_reddit_strong_related.csv")

df_result = await run_sentiment_pipeline_async(df)

df_result.to_csv("dogecoin_sentiment_deepseek.csv", index=False)
print("Done & saved sentiment results.")

🧠 Processing 40377 emotional texts via DeepSeek API...


  2%|▊                                      | 44/2019 [03:56<2:52:41,  5.25s/it]/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/aiohttp/client_reqrep.py:961: RuntimeWarning: coroutine 'batch_process' was never awaited
  with self._timer:
100%|█████████████████████████████████████| 2019/2019 [2:36:29<00:00,  4.65s/it]


✅ Done & saved sentiment results.


In [ ]:
import os
import nest_asyncio
import asyncio
import aiohttp
import pandas as pd
from tqdm import tqdm
import json
import re

nest_asyncio.apply()

# DeepSeek API
API_KEY = os.environ["DEEPSEEK_API_KEY"]
API_URL = "https://api.deepseek.com/v1/chat/completions"

# Prompt

def make_prompt_continuous(text):

    return f"""
You are an emotion analysis model.

Given the text below, first determine whether it is factual/news-like or emotional:

- If factual/news-like, set "fact" = 1, "sentiment_score" = 0, "arousal" = 0.
- If emotional, set "fact" = 0 and assign:
  "sentiment_score": A real number between -1 and 1 (-1 = strongly negative, 0 = neutral, +1 = strongly positive)
  "arousal": A real number between 0 and 1 representing emotional intensity.

Output a JSON object with keys: "fact", "sentiment_score", "arousal".

Text: "{text}"
"""

# DeepSeek
async def analyze_text(session, text):
    try:
        payload = {
            "model": "deepseek-chat",
            "messages": [{"role": "user", "content": make_prompt_continuous(text)}],
            "temperature": 0
        }
        async with session.post(API_URL, headers={"Authorization": f"Bearer {API_KEY}"}, json=payload) as resp:
            result = await resp.json()
            reply = result["choices"][0]["message"]["content"]

            match = re.search(r"\{.*\}", reply, re.S)
            if match:
                data = json.loads(match.group(0))
            else:
                data = {"sentiment_score": 0, "arousal": 0, "error": "parse_fail"}

            return data
    except Exception as e:
        return {"sentiment_score": 0, "arousal": 0, "error": str(e)}

# ========== 批处理 ==========
async def batch_process(texts, batch_size=20):
    results = []
    async with aiohttp.ClientSession() as session:
        for i in tqdm(range(0, len(texts), batch_size)):
            batch = texts[i:i+batch_size]
            tasks = [analyze_text(session, t) for t in batch]
            batch_results = await asyncio.gather(*tasks)
            results.extend(batch_results)
            await asyncio.sleep(2)
    return results

# ========== 主流程 ==========
async def run_sentiment_pipeline_async(df):
    print(f"🧠 Processing {len(df)} texts via DeepSeek API...")
    results = await batch_process(df['text'].tolist())

    df_results = pd.DataFrame(results)
    df_final = pd.concat([df.reset_index(drop=True), df_results], axis=1)
    return df_final

# ========== 执行 ==========
df = pd.read_csv("dogecoin_sentiment_cleaned.csv")

df_result = await run_sentiment_pipeline_async(df)

df_result.to_csv("dogecoin_sentiment_continuous.csv", index=False)
print("✅ Done: Continuous sentiment scoring complete.")


🧠 Processing 40376 texts via DeepSeek API...


100%|█████████████████████████████████████| 2019/2019 [1:45:01<00:00,  3.12s/it]


✅ Done: Continuous sentiment scoring complete.


In [ ]:
async def analyze_text(session, text):
    try:
        payload = {
            "model": "deepseek-chat",
            "messages": [{"role": "user", "content": make_prompt(text)}],
            "temperature": 0
        }
        async with session.post(API_URL, headers={"Authorization": f"Bearer {API_KEY}"}, json=payload) as resp:
            result = await resp.json()

            # ✅ 打印返回结果方便调试
            if "choices" not in result:
                return {
                    "sentiment": "error",
                    "arousal": None,
                    "error": result.get("error", result)  # 保留完整错误信息
                }

            reply = result["choices"][0]["message"]["content"]
            # 提取 JSON
            match = re.search(r"\{.*\}", reply, re.S)
            if not match:
                return {"sentiment": "error", "arousal": None, "error": f"Bad reply: {reply}"}

            data = json.loads(match.group(0))
            return data

    except Exception as e:
        return {"sentiment": "error", "arousal": None, "error": str(e)}


In [ ]:
pip install openai nest_asyncio tqdm pandas aiohttp


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip3.11 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import pandas as pd
from openai import OpenAI
import json
from tqdm import tqdm
import time

# === DeepSeek API 设置 ===
# set DEEPSEEK_API_KEY in your environment before running
client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com"
)

def make_prompt(text):
    return f"""
You are an emotion analysis model.
Given the text below, return a JSON object:
{{
  "sentiment_score": real number in [-1,1],
  "arousal": real number in [0,1]
}}
If the text is factual or neutral, use:
{{"sentiment_score": 0, "arousal": 0}}

Text: "{text}"
"""

# === 分析函数 ===
def analyze_first_100(texts):
    results = []
    for i, text in enumerate(tqdm(texts[:100], desc="Testing first 100 texts")):
        try:
            response = client.chat.completions.create(
                model="deepseek-chat",
                messages=[
                    {"role": "system", "content": "You are a precise emotional scoring model."},
                    {"role": "user", "content": make_prompt(text)}
                ],
                timeout=60
            )
            reply = response.choices[0].message.content
            data = json.loads(reply[reply.find("{"): reply.rfind("}") + 1])
        except Exception as e:
            data = {"sentiment_score": 0, "arousal": 0, "error": str(e)}
        results.append(data)
        time.sleep(0.5)  # 给 API 一点缓冲，避免速率限制
    return results


# === 加载数据 ===
df = pd.read_csv("dogecoin_sentiment.csv")
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"] != ""]
df = df.dropna(subset=["text"])

texts = df["text"].tolist()
print(f"🧠 Total texts: {len(texts)} (processing first 100 only)")

# === 运行分析 ===
results = analyze_first_100(texts)

# === 保存结果 ===
df_results = pd.DataFrame(results)
df_test = pd.concat([df.head(100).reset_index(drop=True), df_results], axis=1)
df_test.to_csv("dogecoin_sentiment_test100.csv", index=False)

print("✅ Done: saved test results to dogecoin_sentiment_test100.csv")


🧠 Total texts: 41590 (processing first 100 only)


Testing first 100 texts: 100%|████████████████| 100/100 [04:48<00:00,  2.89s/it]

✅ Done: saved test results to dogecoin_sentiment_test100.csv


In [ ]:
import os
import pandas as pd
from openai import AsyncOpenAI
import nest_asyncio, asyncio, json, time
from tqdm import tqdm

nest_asyncio.apply()

# DeepSeek API
# set DEEPSEEK_API_KEY in your environment before running
client = AsyncOpenAI(api_key=os.environ["DEEPSEEK_API_KEY"], base_url="https://api.deepseek.com")

# Prompt 
def make_prompt(text):
    return f"""
You are an emotion analysis model.
Given the text below, return a JSON object:
{{
  "sentiment_score": real number in [-1,1],
  "arousal": real number in [0,1],
  "type": one of ["factual","neutral","emotional"]
}}
If type is factual or neutral, set sentiment_score=0 and arousal=0.

Text: "{text}"
"""

# async
async def analyze_text(text, semaphore):
    async with semaphore:
        try:
            response = await client.chat.completions.create(
                model="deepseek-chat",
                messages=[
                    {"role": "system", "content": "You are a precise emotional scoring model."},
                    {"role": "user", "content": make_prompt(text)}
                ],
                timeout=60
            )
            reply = response.choices[0].message.content
            data = json.loads(reply[reply.find("{"):reply.rfind("}")+1])

            t = data.get("type","").lower()
            if t == "factual":
                data["dummy"] = 1
                data["sentiment_score"] = 0
                data["arousal"] = 0
            else:
                data["dummy"] = 0
                if t == "neutral":
                    data["sentiment_score"] = 0
                    data["arousal"] = 0
            data.pop("type", None)
        except Exception as e:
            data = {"sentiment_score":0,"arousal":0,"dummy":0,"error":str(e)}
        return data

#all async
async def analyze_all_async(texts, max_concurrent=10):
    semaphore = asyncio.Semaphore(max_concurrent)
    tasks = [analyze_text(t, semaphore) for t in texts]
    results = []
    for fut in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc="Processing"):
        results.append(await fut)
    return results

#load the data
df = pd.read_csv("dog.csv")
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"] != ""].dropna(subset=["text"])
texts = df["text"].tolist()
print(f"🧠 Total texts: {len(texts)}")

# main running
async def main():
    start = time.time()
    df_results = await analyze_all_async(texts, max_concurrent=10)
    df_final = pd.concat([df.reset_index(drop=True), pd.DataFrame(df_results)], axis=1)
    df_final.to_csv("dog2.csv", index=False)
    print(f"✅ Done! Saved to dogecoin_2c.csv | ⏱ {time.time()-start:.1f}s")

await main()


🧠 Total texts: 2007


Processing: 100%|███████████████████████████| 2007/2007 [08:19<00:00,  4.02it/s]

✅ Done! Saved to dogecoin_2c.csv | ⏱ 499.6s
